# Use Case — Pre-Design Thermal Assessment for a Commercial Office Tower

**Who this is for**  
Building envelope engineers, HVAC designers, and sustainability consultants who need a *measured* microclimate baseline before freezing envelope specs, glazing selection, or cooling capacity.

**The scenario**  
You have been retained to evaluate the thermal performance of a planned commercial office tower in lower Manhattan. Design decisions worth millions hinge on assumptions about the site's thermal environment — ambient temperature, solar load, surrounding surfaces, adjacent shading. Instead of relying on TMY weather files and generic urban context, this notebook replaces each assumption with site-specific data in six steps.

**What you walk away with**
1. A spatial temperature baseline across the site and surroundings
2. Hourly load-driving parameters at the exact coordinates (heat index, wet-bulb, solar GHI/DNI/DHI)
3. Surface composition of the neighborhood from a satellite tile
4. A facade-level view of adjacent shading and sky exposure
5. A consolidated PDF Heat Intelligence Report for the design basis
6. A diurnal swing characterization for HVAC control strategy

Every number you produce here feeds a real engineering deliverable — the energy model, the cooling load calc, the envelope spec, or the client report.

---

## Setup

Load the `.env` from the repo root, instantiate the client, and confirm the key works. If this cell fails, run `notebooks/00_setup.ipynb` first.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from fortyguard import FortyGuardClient
from fortyguard.samples import MANHATTAN_POLYGON

client = FortyGuardClient()

# Design day context used throughout this notebook.
SITE_LAT, SITE_LON = 40.7128, -74.0060          # site coordinates
DESIGN_DATE       = '2024-07-15'                # design-peak summer day
DESIGN_HOUR       = '14:00'                     # design-peak afternoon
DESIGN_TEMP_C     = 32.5                        # expected ambient at peak hour

print(f'Authenticated to {client.base_url}')
print(f'Site: ({SITE_LAT}, {SITE_LON})  •  design day: {DESIGN_DATE} {DESIGN_HOUR}')

---
## Step 1 — Spatial thermal context around the site

### What you are doing
Pulling a high-resolution heatmap across a polygon that covers your site *and* the surrounding blocks, at the design-peak hour. The response contains per-tile temperatures and aggregated statistics.

### Why an engineer cares
Your facades will exchange long-wave radiation with adjacent rooftops, streets, and buildings — not just with a clear sky. A pre-design heatmap tells you:
- Which orientations face the hottest surroundings (higher radiant load on the facade)
- Whether the site sits inside a heat-island cluster, and how much hotter it runs than TMY assumptions
- Where mean-radiant-temperature (MRT) at street level will be uncomfortable for occupants entering the lobby

This is the single most common assumption in energy models that gets silently wrong — and it is the cheapest one to fix.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=MANHATTAN_POLYGON,
    start_date=DESIGN_DATE,
    start_time=DESIGN_HOUR,
    filter_type=1,           # single hour
    granularity=80,          # 80 m resolution
)

stats = heatmap['result'].get('stats_data', {})
t_stats = stats.get('Temperature_stats') or stats.get('temperature_stats') or {}
print(f"Activity ID: {heatmap['activity_id']}")
print('Temperature statistics across the polygon:')
for k, v in t_stats.items():
    print(f'  {k:>22}: {v}')

In [ ]:
# Visualize the thermal gradient across the site.
import folium
import matplotlib.pyplot as plt

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

temps = [f['properties'].get('temperature') for f in features if 'temperature' in f.get('properties', {})]
if temps:
    plt.figure(figsize=(8, 2.8))
    plt.hist(temps, bins=30, color='tomato', edgecolor='white')
    plt.axvline(sum(temps)/len(temps), color='black', linestyle='--', label=f'mean = {sum(temps)/len(temps):.1f} °C')
    plt.xlabel('Tile temperature (°C)'); plt.ylabel('Tile count')
    plt.title('Spatial temperature distribution across the site polygon')
    plt.legend(); plt.tight_layout(); plt.show()

if features:
    lo, hi = min(temps), max(temps)
    def _style(feat):
        t = feat['properties'].get('temperature', lo)
        frac = 0 if hi == lo else (t - lo) / (hi - lo)
        r, b = int(255*frac), int(255*(1-frac))
        return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.65, 'weight': 0}
    fmap = folium.Map(location=[SITE_LAT, SITE_LON], zoom_start=15, tiles='cartodbpositron')
    folium.GeoJson(map_data, style_function=_style).add_to(fmap)
    fmap

### What this tells you
- **Mean vs max tile temperature** — the delta is the intra-site gradient you cannot see in weather-station data.
- **Map pattern** — hot clusters upstream of prevailing wind are a direct indicator that your facade will see ambient warmer than the TMY.

**Engineering takeaways**
- Site ambient more than ~2 °C above the nearest weather station → override TMY dry-bulb in your energy model.
- Hottest adjacent block on the W or SW side → do not commit to single glazing on that orientation without shading.
- Gradient across the footprint → tower placement on the site can shift mean solar exposure significantly.

---
## Step 2 — Thermal load drivers at the exact point

### What you are doing
Querying environmental parameters at the site coordinates across the design day (09:00 → 17:00 hourly). The response bundles heat index, apparent temperature, wet-bulb temperature, relative humidity, and three solar irradiance components (GHI / DNI / DHI).

### Why an engineer cares
These are the raw numbers that feed every cooling-load calculation you will do on this project:
- **Wet-bulb temperature** sizes cooling towers and tells you whether direct or indirect evaporative cooling is viable
- **GHI / DNI / DHI** drive solar gain through glazing and determine PV feasibility; they also differentiate *direct* from *diffuse* shading requirements
- **Heat index** is what occupants feel — you need it for comfort SLAs and lobby-entrance design
- **Relative humidity** governs the latent-vs-sensible split in the cooling load

In [ ]:
env = client.environmental_parameters(
    latitude=SITE_LAT,
    longitude=SITE_LON,
    temperature=DESIGN_TEMP_C,
    start_date=DESIGN_DATE,
    start_time='09:00',
    end_time='17:00',
    filter_type=2,    # range of hours
)

result   = env['result']
location = result['locations'][0]
params   = location.get('parameters', {})
solar    = location.get('solar_irradiance', {}).get('clear_sky', {})

print('Clear-sky solar irradiance (W/m²) at design hour:')
for comp in ('ghi', 'dni', 'dhi'):
    print(f'  {comp.upper():>5}: {solar.get(comp)}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

timestamps = result['metadata'].get('timestamps', [])
df = pd.DataFrame({k: v for k, v in params.items()
                   if isinstance(v, list) and len(v) == len(timestamps)})
df.insert(0, 'timestamp', pd.to_datetime(timestamps))
df.set_index('timestamp', inplace=True)

thermal = [c for c in ('apparent_temperature_celsius',
                       'heat_index_celsius',
                       'wet_bulb_temperature_celsius',
                       'relative_humidity_percent') if c in df.columns]
if thermal:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
    df[[c for c in thermal if 'humidity' not in c]].plot(ax=ax1, marker='o')
    ax1.set_title('Apparent / heat index / wet-bulb across the design day')
    ax1.set_ylabel('°C'); ax1.grid(alpha=0.3)
    if 'relative_humidity_percent' in df.columns:
        df['relative_humidity_percent'].plot(ax=ax2, color='steelblue', marker='o')
        ax2.set_title('Relative humidity'); ax2.set_ylabel('%'); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

df.head()

### What this tells you
- The hour-by-hour thermal-comfort and solar profile your design day actually delivers.
- Whether your cooling load is sensible-dominated (low RH, high DNI) or latent-dominated (high RH, high wet-bulb).

**Engineering takeaways**
- Peak wet-bulb > 24 °C → evaporative cooling loses efficiency; specify mechanical cooling
- DNI dominant → horizontal overhangs and vertical fins pay off; diffuse-dominant days need different geometry
- Heat index ≥ 32 °C for 5+ hours → conservative indoor setpoints and lobby vestibules are warranted

---
## Step 3 — Surrounding surface composition

### What you are doing
Running satellite segmentation on the tile centered at the site. The API classifies every pixel of the aerial view into categories (rooftops, roads, vegetation, water, bare land, etc.) and returns the coverage percentages.

### Why an engineer cares
Energy models typically assume a generic "urban" or "suburban" context. That assumption is wrong by enough to matter. Knowing, for example, that 72 % of your surroundings is impervious and 4 % is vegetation changes:
- **Local albedo** (higher impervious fraction → more reflected radiation onto lower-floor glazing)
- **Evaporative cooling from nearby vegetation** (affects ambient temperature at the site)
- **Urban heat-island intensity estimate** (feeds the ambient override you applied in Step 1)

In [ ]:
sat = client.satellite_segmentation(
    latitude=SITE_LAT,
    longitude=SITE_LON,
    start_date=DESIGN_DATE,
    start_time=DESIGN_HOUR,
    filter_type=1,
    granularity=80,
)
sat_result = sat['result']
segments   = sat_result.get('segmentation', {}).get('segments', {})

print(f"Image year: {sat_result.get('image_year')}")
print('Surface coverage around the site:')
for cls, pct in sorted(segments.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {cls:>28}: {pct}')

In [ ]:
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

originals = sat_result.get('orignal_image') or sat_result.get('original_image') or []
mask_b64  = sat_result.get('segmentation', {}).get('image_content')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, img, title in zip(axes,
                          [_decode(originals[0]) if originals else None, _decode(mask_b64)],
                          ['Satellite tile', 'Segmentation mask']):
    if img is not None: ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

### What this tells you
- The share of each surface class in the immediate surroundings.
- Where vegetation clusters sit relative to your site — and therefore which orientations benefit most from evaporative cooling.

**Engineering takeaways**
- Impervious + rooftop fraction > 65 % → expect a 1 – 3 °C ambient bump over TMY; the Step 1 heatmap should confirm this.
- Vegetation fraction < 10 % → do *not* take credit for tree-assisted cooling in the energy model.
- If a "green corridor" exists on one side, orient operable windows or outdoor-air intakes toward it.

---
## Step 4 — Facade-level exposure

### What you are doing
Running street view segmentation at the site, oriented toward the primary facade. The API returns a ground-level image plus a pixel-wise segmentation into buildings, sky, vegetation, roads, and so on — as well as the coverage percentages.

### Why an engineer cares
The satellite view does not show you what is *visible from the window*. Street view does, and that directly drives:
- **External shading feasibility** — do adjacent buildings already shade the facade at design hour?
- **Daylighting performance** — the sky fraction from the window dictates useful daylight illuminance
- **Long-wave radiant exchange** — the sky fraction also sets the night-time cooling potential of the facade
- **Ground reflectance** — bright pavement visible below the window amplifies lower-pane solar gain

In [ ]:
street = client.street_view_segmentation(
    latitude=SITE_LAT,
    longitude=SITE_LON,
    vertical_angle=10.0,
    horizontal_angle=270.0,   # facing west — primary summer afternoon exposure
    back_view=False,
)
street_result = street['result']
front = street_result.get('front', {})
seg_pct = front.get('segments', {})

print('Facade sight-line coverage (facing west):')
for cls, pct in sorted(seg_pct.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {cls:>20}: {pct}')

In [ ]:
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [(axes[0], 'original_image',   'Street view (west-facing)'),
                       (axes[1], 'segmented_image',  'Segmentation')]:
    img = _decode(front.get(key))
    if img is not None: ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

### What this tells you
- **Sky fraction** — determines diffuse daylight potential and long-wave to-sky cooling.
- **Adjacent-building fraction** — how much of the facade is already self-shaded by surroundings.
- **Ground / vegetation fraction** — reflectance into lower floors and any cooling-island effect nearby.

**Engineering takeaways**
- Sky fraction > 40 % → external shading is required on this orientation; tower self-shading alone is insufficient.
- Adjacent-building fraction > 60 % during afternoon hours → SHGC spec can be relaxed here.
- Ground-level vegetation < 5 % → no microclimate cooling benefit at street-level entries; specify canopy or water feature if occupant comfort is in scope.

---
## Step 5 — Consolidated Heat Intelligence Report

### What you are doing
Requesting a Heat Intelligence Report, which stitches geographic, environmental, and urban dimensions into a single PDF. Submit the job and the client streams the finished report into `outputs/`.

### Why an engineer cares
Every envelope decision needs documented justification — for the client, the sustainability consultant, or a LEED / BREEAM submittal. The report consolidates everything you have inspected in Steps 1 – 4 into one PDF that lives in the Design Basis Document. It saves you half a day of screenshot-wrangling and eliminates the version-confusion that creeps in when the team revises.

In [ ]:
from IPython.display import FileLink

pdf_path = client.heat_intelligence(
    latitude=SITE_LAT,
    longitude=SITE_LON,
    temperature=DESIGN_TEMP_C,
    date=DESIGN_DATE,
    analysis=['environmental', 'urban', 'geographic'],
)
print(f'Saved to: {pdf_path.resolve()}')
FileLink(str(pdf_path))

### What this gives you
- A PDF deliverable ready to attach to the Design Basis Document.
- A cross-dimension narrative the client can read without technical training — useful for budget sign-off.
- A stable reference you can regenerate with different inputs to document design iterations.

---
## Step 6 — Diurnal swing characterization

### What you are doing
Pulling the same polygon at two points in the design day — a pre-dawn minimum (05:00) and the afternoon peak (14:00) — and comparing the stats. The delta between the two is the **diurnal temperature range (DTR)** that your HVAC system and thermal mass have to handle.

### Why an engineer cares
A 14 °C DTR and a 4 °C DTR demand completely different design strategies:
- **High DTR (>10 °C)** → night-flushing + exposed thermal mass becomes viable passive cooling
- **Low DTR (<5 °C)** → thermal-mass payback is weak; invest in active controls instead
- Either way, the worst-case DTR sizes the cooldown/reheat cycles that fatigue the HVAC equipment

In [ ]:
# Afternoon peak — we already have this from Step 1; run the pre-dawn minimum.
predawn = client.create_heatmap(
    polygon_aoi=MANHATTAN_POLYGON,
    start_date=DESIGN_DATE,
    start_time='05:00',
    filter_type=1,
    granularity=80,
)

def _extract(hm):
    s = hm['result'].get('stats_data', {})
    return s.get('Temperature_stats') or s.get('temperature_stats') or {}

peak_stats    = _extract(heatmap)
predawn_stats = _extract(predawn)

def _g(d, *keys):
    for k in keys:
        if k in d: return d[k]
    return None

peak_mean    = _g(peak_stats,    'mean', 'Mean')
predawn_mean = _g(predawn_stats, 'mean', 'Mean')
peak_max     = _g(peak_stats,    'max',  'Max')
predawn_min  = _g(predawn_stats, 'min',  'Min')

dtr_mean = peak_mean - predawn_mean if (peak_mean and predawn_mean) else None
dtr_abs  = peak_max  - predawn_min  if (peak_max  and predawn_min)  else None

print(f'Pre-dawn (05:00)  mean: {predawn_mean}  min: {predawn_min}')
print(f'Peak    (14:00)  mean: {peak_mean}     max: {peak_max}')
print('-' * 42)
print(f'Mean diurnal range : {dtr_mean} °C')
print(f'Absolute swing     : {dtr_abs} °C')

In [ ]:
import matplotlib.pyplot as plt

labels = ['Mean', 'Min / Max']
predawn_vals = [predawn_mean or 0, predawn_min or 0]
peak_vals    = [peak_mean    or 0, peak_max    or 0]

x = range(len(labels))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([i - 0.2 for i in x], predawn_vals, width=0.4, label='Pre-dawn (05:00)', color='steelblue')
ax.bar([i + 0.2 for i in x], peak_vals,    width=0.4, label='Peak (14:00)',    color='tomato')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('°C'); ax.set_title('Diurnal temperature comparison across the site')
ax.legend(); ax.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

### What this tells you
- The diurnal temperature range your building actually experiences, not a generic climate-zone value.
- Whether DTR varies across the polygon — shaded zones stay warmer overnight because they radiate to the sky less.

**Engineering takeaways**
- DTR > 10 °C → night-flushing through operable vents is likely viable; plan for it at the envelope stage (operable facade, thermal mass exposure).
- DTR < 5 °C → rely on mechanical cooling; thermal-mass strategies will not earn back their cost.
- Site-internal DTR variance → inform where to place air intakes (colder side = better free cooling window).

---
## Wrap-up — what you now have in hand

Running this notebook end-to-end produces six deliverables that plug directly into the project's engineering workflow:

| # | Deliverable | Feeds |
|---|-------------|-------|
| 1 | Spatial temperature baseline | TMY override in the energy model; facade orientation review |
| 2 | Hourly wet-bulb / GHI / DNI / DHI profile | Cooling load calc; cooling tower sizing; glazing SHGC selection |
| 3 | Surrounding surface composition | Local albedo and UHI intensity assumptions |
| 4 | Facade sight-line segmentation | External shading design; daylighting performance |
| 5 | Heat Intelligence PDF | Design Basis Document; LEED / BREEAM submittal support |
| 6 | Diurnal swing characterization | HVAC control sequence; thermal-mass strategy |

Every number came from the site, the hour, and the fabric you actually design into — not from a TMY file and a rule of thumb. That is the difference between a report the client files and a report the client *quotes*.

### Where to go next
- Swap `SITE_LAT` / `SITE_LON` / `MANHATTAN_POLYGON` to your own project and rerun the notebook.
- Iterate Step 4 across several `horizontal_angle` values (0, 90, 180, 270) to document each facade.
- Re-run Step 6 for the winter design day to verify heating-season DTR and overnight setback strategy.